**A. Extract: Membaca Data dari HDFS**

In [4]:
from pyspark.sql import SparkSession 
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number 
from pyspark.sql.window import Window 

spark = SparkSession.builder \
    .appName("Tugas7-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate() 

spark.sparkContext.setLogLevel("ERROR") 

df_transaksi = spark.read.csv( 
    
"hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv", 
    header=True, 
    inferSchema=True 
) 

df_cabang = spark.read.json( 
    
"hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json" 
) 

df_menu = spark.read.csv( 
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv", 
    header=True, 
    inferSchema=True 
) 

print("Jumlah transaksi:", df_transaksi.count()) 
print("Jumlah cabang:", df_cabang.count()) 
print("Jumlah menu:", df_menu.count())

Jumlah transaksi: 4000
Jumlah cabang: 3
Jumlah menu: 15


**B. Transform: Join dan Kolom Turunan**

In [5]:
df_gabungan = df_transaksi.join(
    df_cabang,
    on="cabang_id",
    how="inner"
).join(
    df_menu,
    on="menu_id",
    how="inner"
)

df_gabungan = df_gabungan.withColumn(
    "total_penjualan",
    col("qty") * col("harga")
)

df_gabungan.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


**C. Transform: Top-2 Menu Terlaris pada Setiap Cabang**

In [6]:
ringkasan_menu_cabang = df_gabungan.groupBy(
    "nama_cabang", "nama_menu"
).agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

window_cabang = Window.partitionBy(
    "nama_cabang"
).orderBy(
    col("total_penjualan").desc()
)

top2_menu = ringkasan_menu_cabang.withColumn(
    "peringkat",
    row_number().over(window_cabang)
).filter(
    col("peringkat") <= 2
)

top2_menu.orderBy("nama_cabang", "peringkat").show()

+--------------------+---------+---------------+---------+
|         nama_cabang|nama_menu|total_penjualan|peringkat|
+--------------------+---------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|        8160000|        2|
+--------------------+---------+---------------+---------+



**D. Transform: Total Penjualan per Kepala Barista Menggunakan Spark SQL**

In [7]:
df_gabungan.createOrReplaceTempView("penjualan_kopi")

hasil_barista = spark.sql("""
    SELECT
        kepala_barista,
        SUM(total_penjualan) AS total_penjualan
    FROM penjualan_kopi
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC 
""")

hasil_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



**E. Load: Menyimpan Data Gabungan ke HDFS dalam Format Parquet**

In [8]:
df_gabungan.write.mode("overwrite").partitionBy(
    "kategori_menu"
).parquet(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"
)

print("Data berhasil disimpan dalam format Parquet.")

df_verifikasi = spark.read.parquet(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"
)

print("Jumlah baris hasil penyimpanan:", df_verifikasi.count())
df_verifikasi.show(5)

Data berhasil disimpan dalam format Parquet.
Jumlah baris hasil penyimpanan: 4000
+-------+---------+------+---+--------------+----------+--------------------+---------+-----+---------------+--------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu|harga|total_penjualan| kategori_menu|
+-------+---------+------+---+--------------+----------+--------------------+---------+-----+---------------+--------------+
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|35000|         140000|Makanan Ringan|
|      9|        3|   KN8|  4|         Nadia|  Semarang|Kopi Nusantara Se...|   Menu-9|35000|         140000|Makanan Ringan|
|      3|        1|  KN10|  1|         Nadia|  Magelang|Kopi Nusantara Ma...|   Menu-3|30000|          30000|Makanan Ringan|
|     13|        2|  KN13|  3|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-13|35000|         105000|Makanan Ringan|
|     12|        3|  KN15|  1|         Nadi

**F. Ringkasan Eksekutif dan Rekomendasi**

Kode ringkasan eksekutif:

In [9]:
ringkasan_eksekutif = df_verifikasi.groupBy(
    "nama_cabang", "kota", "kepala_barista"
).agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(
    col("total_penjualan").desc()
)

ringkasan_eksekutif.show()

+--------------------+----------+--------------+---------------+----------------+-------------------------+
|         nama_cabang|      kota|kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Se...|  Semarang|         Nadia|       90542000|            1393|         64997.8463747308|
|Kopi Nusantara Ma...|  Magelang|         Nadia|       87138000|            1302|        66926.26728110599|
|Kopi Nusantara Yo...|Yogyakarta|          Reza|       84645000|            1305|        64862.06896551724|
+--------------------+----------+--------------+---------------+----------------+-------------------------+



**Analisis dan rekomendasi:**

Berdasarkan hasil ringkasan eksekutif, cabang dengan total penjualan tertinggi dapat diidentifikasi sebagai cabang dengan kinerja penjualan terbaik selama periode pengamatan. Sebaliknya, cabang dengan total penjualan terendah perlu mendapatkan perhatian lebih dalam evaluasi operasional dan strategi pemasaran. Untuk menentukan menu yang perlu dipromosikan pada cabang dengan kinerja terlemah, hasil top-2 menu pada bagian C dapat dijadikan acuan. Menu yang memiliki nilai penjualan tertinggi di cabang tersebut dapat diprioritaskan dalam promosi, misalnya melalui paket bundling, diskon pada jam tertentu, atau rekomendasi menu pada media sosial. Strategi ini bertujuan meningkatkan visibilitas menu dan mendorong pembelian pelanggan. Selain itu, jumlah transaksi dan rata-rata nilai penjualan perlu diperhatikan agar evaluasi tidak hanya berfokus pada total pendapatan. Rekomendasi akhir harus disesuaikan dengan hasil pengolahan data yang benar-benar muncul setelah seluruh kode dijalankan, sehingga keputusan yang diambil berdasarkan temuan data dan bukan sekadar perkiraan.